# Task5 — Directional gap-cut

## 1. Setup และเส้นทาง
Dependencies: numpy, pandas, scipy, scikit-learn, numba, joblib, pyarrow. ใช้kernel .venv เดิม

In [ ]:
from pathlib import Path
import json,hashlib,time
import numpy as np
import pandas as pd
import joblib
from numba import njit
from scipy.stats import spearmanr
from sklearn.isotonic import IsotonicRegression
from IPython.display import display
# Locate Task5 directly; no other notebook or project module is required.
candidates=[q for p in (Path.cwd(),*Path.cwd().parents) for q in (p,p/'Task5',p/'CPE342-Kaggle/Task5')]
TASK=next((p for p in candidates if (p/'data/raw/test.csv').exists() and p.name=='Task5'),None)
if TASK is None:raise FileNotFoundError('Open this notebook inside Task5, or set TASK to the folder containing data/raw/test.csv')
RAW=TASK/'data/raw/test.csv'
OUT=TASK/'models/results/directional_gap_cut_standalone_v1'
CONFIG={'seeds':[173,271,389],'partition_repeats':64,'sample_sizes':[256,1024],'budget':2046}
EXPECTED_RAW_SHA='0b9d42b8129e09825f73222fcbf5f93967e98242f5ec107bb2cb9d83ccf4e148'
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def write_json(p,obj):
    p=Path(p);tmp=p.with_suffix('.partial.json');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False)+'\n');tmp.replace(p)
def top(x,k):
    assert 0<k<=len(x)
    y=np.zeros(len(x),dtype=np.int8);y[np.argsort(-np.asarray(x),kind='stable')[:k]]=1
    return y
print('Raw:',RAW,'Output:',OUT)

## 2. อ่านข้อมูลและสร้างfamily matrices
Temporal featuresใช้arcsinh; geographyใช้log1pระยะทางHaversineระหว่างช่วงติดกัน. ไม่ตีความเป็นความเร็ว

In [ ]:
FAMILIES=['location_changes', 'device_count', 'ip_hash_entropy', 'login_time_variance', 'login_count', 'session_duration_mean', 'session_duration_std', 'play_pattern_consistency', 'activity_level', 'skill_rating', 'skill_trajectory', 'kda_mean', 'kda_std', 'accuracy_mean', 'accuracy_std', 'reaction_time_mean', 'reaction_time_std', 'performance_stability', 'purchase_frequency', 'purchase_amount_mean', 'purchase_amount_std', 'refund_count', 'currency_transfers', 'friend_adds', 'friend_removes', 'communication_frequency', 'guild_changes', 'geo_steps']

def matrices(raw):
    out={f:np.arcsinh(raw[[f'{f}_{t}' for t in range(1,5)]].to_numpy(float)) for f in FAMILIES if f!='geo_steps'}
    loc=[]
    for t in range(1,5):
        la=raw[f'login_lat_{t}'].to_numpy(float);lo=raw[f'login_lon_{t}'].to_numpy(float)
        ok=np.isfinite(la)&np.isfinite(lo)&(abs(la)<=90)&(abs(lo)<=180)
        loc.append((np.where(ok,np.radians(la),np.nan),np.where(ok,np.radians(lo),np.nan)))
    steps=[]
    for (la,lo),(lb,lp) in zip(loc[:-1],loc[1:]):
        h=np.sin((lb-la)/2)**2+np.cos(la)*np.cos(lb)*np.sin((lp-lo)/2)**2
        steps.append(np.log1p(12742*np.arcsin(np.sqrt(np.clip(h,0,1)))))
    out['geo_steps']=np.column_stack(steps)
    return out
raw=pd.read_csv(RAW)
assert sha(RAW)==EXPECTED_RAW_SHA,'Raw dataset changed; review configuration before running'
assert raw.player_id.is_unique
assert len(raw)>=CONFIG['budget']
print('Rows:',len(raw),'Columns:',len(raw.columns))
display(raw.isna().sum().to_frame('missing_count'))

## 3. Preprocessing และชนิดข้อมูล
เติมmissingในtemporalfamilyด้วยmedianในแถว แล้วfallbackmedianรวม. ใช้robustscale. กลุ่มlow-cardinality integerไม่ใช้numericgap ไม่ได้แปลว่าปกติ

In [ ]:
def temporal_preprocess(raw):
    values=matrices(raw);result={};metadata={}
    for family,a in values.items():
        frame=pd.DataFrame(a);rowmed=frame.median(axis=1).to_numpy()
        median=float(np.nanmedian(a));rowmed=np.where(np.isfinite(rowmed),rowmed,median)
        filled=np.where(np.isfinite(a),a,rowmed[:,None])
        q=np.nanquantile(a,[.25,.75]);scale=float((q[1]-q[0])/1.349)
        if scale<1e-6:scale=max(float(np.nanstd(a)),1.)
        result[family]=np.ascontiguousarray((filled-median)/scale,dtype=np.float64)
        metadata[family]=dict(median=median,scale=scale,missing_fraction=float(np.isnan(a).mean()),all_missing_rows=int(np.isnan(a).all(1).sum()))
    return result,metadata

def preprocess(raw):
    spaces,prep=temporal_preprocess(raw)
    for name,cols in [('static_traits',['account_age_days','behavior_stability']),('security_flags',['password_changed','vpn_usage','suspicious_login_time','mass_item_sale'])]:
        a=raw[cols].to_numpy(float)
        if name=='static_traits':a=np.arcsinh(a)
        med=np.nanmedian(a,axis=0);q=np.nanquantile(a,[.25,.75],axis=0);sd=np.nanstd(a,axis=0)
        scale=np.where(q[1]-q[0]>1e-6,(q[1]-q[0])/1.349,np.where(sd>1e-6,sd,1)) if name=='static_traits' else np.ones(a.shape[1])
        spaces[name]=np.ascontiguousarray((np.where(np.isfinite(a),a,med)-med)/scale)
        prep[name]=dict(columns=cols,median=med.tolist(),scale=scale.tolist(),imputation='column median')
    return spaces,prep

def eligibility(raw):
    a=matrices(raw);records=[]
    for family in preprocess(raw)[0]:
        if family in ['static_traits','security_flags']:
            cols=['account_age_days','behavior_stability'] if family=='static_traits' else ['password_changed','vpn_usage','suspicious_login_time','mass_item_sale'];v=raw[cols].to_numpy(float)
        elif family=='geo_steps':v=np.expm1(a[family])
        else:v=raw[[f'{family}_{i}' for i in range(1,5)]].to_numpy(float)
        uniques=[len(np.unique(c[np.isfinite(c)])) for c in v.T]
        ints=all(np.allclose(c[np.isfinite(c)],np.round(c[np.isfinite(c)]),atol=1e-10,rtol=0) for c in v.T)
        categorical=bool(max(uniques)<=10 and ints)
        records.append(dict(family=family,distinct_values=uniques,integer_valued=bool(ints),included=not categorical,reason='continuous numeric-gap score' if not categorical else 'category spacing is not continuous separation'))
    return records
spaces,preparation=preprocess(raw)
types=eligibility(raw)
retained=[x['family'] for x in types if x['included']]
assert len(retained)==26
assert all(np.isfinite(x).all() for x in spaces.values())
display(pd.DataFrame(types))

## 4. Random cuts และgap-weighted collusive displacement
Static range-weighted partitions. Relative gapเป็นการดัดแปลงของการทดลองนี้ ไม่ใช่theoremของRRCF. ใช้Numbaเพื่อให้CPUคำนวณเร็วขึ้น

In [ ]:
@njit(cache=False)
def gap_cut_scores(x,seed):
    """Build a complete RRCT; duplicate points share one leaf.
    Score=max sibling_mass/subtree_mass over ancestors for each sampled point.
    Same tree also returns an isolation path-depth control.
    """
    np.random.seed(seed);n,d=x.shape
    order=np.arange(n);lo=np.zeros(2*n,dtype=np.int64);hi=np.zeros(2*n,dtype=np.int64)
    parent=np.full(2*n,-1,dtype=np.int64);left=np.full(2*n,-1,dtype=np.int64);right=left.copy();depth=np.zeros(2*n,dtype=np.int64)
    gaps=np.zeros(2*n)
    stack=np.zeros(2*n,dtype=np.int64);stack[0]=0;sp=1;hi[0]=n;nodes=1
    while sp>0:
        sp-=1;node=stack[sp];a=lo[node];b=hi[node]
        if b-a<=1:continue
        mins=np.full(d,np.inf);maxs=np.full(d,-np.inf)
        for k in range(a,b):
            row=order[k]
            for j in range(d):
                value=x[row,j]
                if value<mins[j]:mins[j]=value
                if value>maxs[j]:maxs[j]=value
        lengths=maxs-mins;total=lengths.sum()
        if total<=1e-12:continue
        pick=np.random.random()*total;dim=0
        while dim<d-1 and pick>=lengths[dim]:pick-=lengths[dim];dim+=1
        cut=mins[dim]+np.random.random()*lengths[dim]
        # Make the cut strictly below max to ensure both children are nonempty.
        if cut>=maxs[dim]:cut=np.nextafter(maxs[dim],mins[dim])
        i=a;j=b-1
        while i<=j:
            if x[order[i],dim]<=cut:i+=1
            else:tmp=order[i];order[i]=order[j];order[j]=tmp;j-=1
        mid=i
        if mid==a or mid==b:continue
        lmax=-np.inf;rmin=np.inf
        for k in range(a,mid):lmax=max(lmax,x[order[k],dim])
        for k in range(mid,b):rmin=min(rmin,x[order[k],dim])
        gaps[node]=max(0.,rmin-lmax)/lengths[dim]
        l=nodes;r=nodes+1;nodes+=2;left[node]=l;right[node]=r
        lo[l]=a;hi[l]=mid;lo[r]=mid;hi[r]=b
        parent[l]=node;parent[r]=node;depth[l]=depth[node]+1;depth[r]=depth[node]+1
        stack[sp]=l;stack[sp+1]=r;sp+=2
    codisp=np.zeros(n);path=np.zeros(n);gapscore=np.zeros(n)
    for node in range(nodes):
        if left[node]>=0:continue
        best=0.;gscore=0.;at=node
        while parent[at]>=0:
            p=parent[at];sibling=right[p] if left[p]==at else left[p]
            ratio=(hi[sibling]-lo[sibling])/(hi[at]-lo[at])
            if ratio>best:best=ratio
            gscore=max(gscore,ratio*gaps[p])
            at=p
        mass=hi[node]-lo[node];extra=0.
        if mass>1:
            if mass==2:extra=1.
            else:extra=2*(np.log(mass-1)+.5772156649015329)-2*(mass-1)/mass
        for k in range(lo[node],hi[node]):
            codisp[order[k]]=best;path[order[k]]=depth[node]+extra;gapscore[order[k]]=gscore
    return codisp,path,gapscore

def family_scores(x,seed,repeats=None):
    repeats=CONFIG['partition_repeats'] if repeats is None else repeats
    n=len(x);rng=np.random.default_rng(seed);cd=np.zeros(n);gs=np.zeros(n);seen=np.zeros(n)
    for size in CONFIG['sample_sizes']:
        chunks=int(np.ceil(n/size))
        for repeat in range(repeats):
            order=rng.permutation(n)
            for rows in np.array_split(order,chunks):
                c,_,g=gap_cut_scores(x[rows],int(rng.integers(1,2**31-1)))
                cd[rows]+=np.log1p(c);gs[rows]+=np.log1p(g);seen[rows]+=1
    assert (seen==repeats*len(CONFIG['sample_sizes'])).all()
    return cd/seen,gs/seen

def aggregate(a):return np.sort(a,axis=1)[:,-2:].mean(axis=1)

## 5. ตรวจสูตรและคำนวณ3bags
เก็บcacheของเล่มนี้เท่านั้น. seedoffsetคงลำดับfamilyเดิมแม้ข้ามกลุ่มที่ไม่ใช้ เพื่อรักษาสูตร. กดInterruptแล้วRun Allต่อได้; familyที่ค้างคำนวณใหม่

In [ ]:
c,h,g=gap_cut_scores(np.zeros((20,3)),4)
assert (g==0).all()
z=np.r_[np.zeros((200,2)),np.full((4,2),10.)]
c,h,g=gap_cut_scores(z,1);assert np.allclose(c,g)
OUT.mkdir(parents=True,exist_ok=True)
notebook_path=TASK/'Task5_cut_study.ipynb'
notebook=json.loads(notebook_path.read_text())
code_hash=hashlib.sha256('\n'.join(''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code').encode()).hexdigest()
settings={'raw_sha256':sha(RAW),'config':CONFIG,'code_sha256':code_hash,'families':list(spaces),'retained':retained}
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Code/data/config changed: use a new OUT'
else:write_json(OUT/'settings.json',settings)
frames=[]
for seed in CONFIG['seeds']:
    frame=pd.DataFrame({'row_index':np.arange(len(raw)),'seed':seed})
    for j,(family,x) in enumerate(spaces.items()):
        if family not in retained:continue
        p=OUT/f'{family}_seed{seed}.parquet'
        if p.exists():
            cache=pd.read_parquet(p)
            assert np.array_equal(cache.row_index,np.arange(len(raw)))
        else:
            tick=time.monotonic()
            _,gap=family_scores(x,seed+1009*j)
            cache=pd.DataFrame({'row_index':np.arange(len(raw)),'gap':gap})
            temp=p.with_suffix('.partial.parquet');cache.to_parquet(temp,index=False);temp.replace(p)
            print('seed',seed,family,'seconds',round(time.monotonic()-tick,1),flush=True)
        assert np.isfinite(cache.gap).all()
        frame[family+'__gap']=cache.gap.to_numpy()
    frames.append(frame)
long=pd.concat(frames,ignore_index=True)
print('Completed bags:',CONFIG['seeds'])

## 6. Geographic monotone projection และaggregate
ให้geographicscoreเพิ่มตามmaximum adjacent distance ด้วยisotonicregression. เป็นdomain assumption ไม่ใช่probabilitycalibrationหรือหลักฐานfraud. All-missing geographyให้0

In [ ]:
steps=np.expm1(matrices(raw)['geo_steps'])
observed=np.isfinite(steps).any(axis=1)
distance=np.max(np.where(np.isfinite(steps),steps,-np.inf),axis=1)
assert observed.any()
cols=[f+'__gap' for f in retained]
replay=[]
for seed,g in long.groupby('seed',sort=True):
    g=g.sort_values('row_index').copy()
    model=IsotonicRegression(increasing=True,out_of_bounds='clip')
    model.fit(distance[observed],g['geo_steps__gap'].to_numpy()[observed])
    geo=np.zeros(len(raw));geo[observed]=model.predict(distance[observed])
    assert np.all(np.diff(model.predict(np.sort(distance[observed])))>=-1e-12)
    joblib.dump(model,OUT/f'geo_seed{seed}.joblib')
    g['geo_steps__gap']=geo;g['directional_gap']=aggregate(g[cols].to_numpy())
    replay.append(g)
long=pd.concat(replay,ignore_index=True)
long.to_parquet(OUT/'bag_scores.parquet',index=False)
scores=long.groupby('row_index')[['directional_gap',*cols]].mean().reindex(range(len(raw)))
scores.insert(0,'id',[f'ANS{i+1:05d}' for i in range(len(raw))])
scores.insert(1,'player_id',raw.player_id.to_numpy())
scores['prediction']=top(scores.directional_gap,CONFIG['budget'])
assert scores.id.is_unique and np.isfinite(scores.directional_gap).all()
assert int(scores.prediction.sum())==CONFIG['budget']
display(scores[['id','player_id','directional_gap','prediction']].head())

## 7. Diagnosticsของรุ่นนี้เอง
SeedJaccard/rankcorrelationวัดความสม่ำเสมอ ไม่ใช่accuracyหรือF3; ไม่มีreferencepredictions

In [ ]:
stability=[]
bags=[(seed,g.sort_values('row_index')) for seed,g in long.groupby('seed',sort=True)]
for i,(seed_a,a) in enumerate(bags):
    for seed_b,b in bags[i+1:]:
        aa=top(a.directional_gap,CONFIG['budget']).astype(bool);bb=top(b.directional_gap,CONFIG['budget']).astype(bool)
        stability.append({'seed_a':int(seed_a),'seed_b':int(seed_b),'jaccard':float((aa&bb).sum()/(aa|bb).sum()),'spearman':float(spearmanr(a.directional_gap,b.directional_gap).statistic)})
display(pd.DataFrame(stability))
display(scores.groupby('prediction').directional_gap.describe())
report={'rows':len(raw),'alerts':int(scores.prediction.sum()),'budget':CONFIG['budget'],'seed_stability':stability,'measured_task5_F3':None,'raw_unchanged':sha(RAW)==EXPECTED_RAW_SHA}
write_json(OUT/'report.json',report)

## 8. ExportคำตอบTask5
ไฟล์นี้มีเฉพาะid/task5 ไม่ใช่combinedsubmission5Tasks. ไม่อ่านไฟล์จากTaskอื่นและไม่upload

In [ ]:
scores.to_parquet(OUT/'predictions.parquet',index=False)
answer=scores[['id','prediction']].rename(columns={'prediction':'task5'})
path=OUT/'task5_predictions.csv';answer.to_csv(path,index=False)
check=pd.read_csv(path,dtype={'id':str})
assert check.id.tolist()==answer.id.tolist() and np.array_equal(check.task5,answer.task5)
print('Task5 predictions:',path)
print('Standalone run complete; no Kaggle upload performed.')

## References
[Guha et al. 2016 — Robust Random Cut Forest](https://proceedings.mlr.press/v48/guha16.html): random cuts/collusive displacement. สูตรrelative-gap, type filtering และmonotonegeographyในเล่มนี้เป็นส่วนดัดแปลงที่ต้องประเมินแยก ไม่ใช่standardRRCFทั้งหมด